# APOStrategyActorCriticJ

> JAX CRLD actor-critic agents under partial observability. Inherits from POstrategybaseJ.

In [ ]:
#| default_exp Agents/POStrategyActorCriticJ

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

In [ ]:
#| export
import jax
from jax import jit
import jax.numpy as jnp
import itertools as it
from functools import partial
from fastcore.utils import *

from pyCRLD.Agents.POStrategyBaseJ import POstrategybaseJ
from pyCRLD.Utils.HelpersJ import *

In [ ]:
#| export
class POstratACJ(POstrategybaseJ):
    """Class for JAX CRLD actor-critic agents under partial observability in strategy space."""

    @partial(jit, static_argnums=(0, 2))
    def RPEioa(self, X, norm=False):
        """TD error for partially observable actor-critic dynamics, given joint policy X."""
        Bios = self.fast_Bios(X)
        Xisa = self.Xisa(X)

        R = self.Rioa(X, Bios=Bios, Xisa=Xisa)
        Vio = self.Vio(X, Bios=Bios, Xisa=Xisa, Rioa=R)
        NextV = self.NextVioa(X, Bios=Bios, Xisa=Xisa, Vio=Vio)

        n = jnp.newaxis
        E = self.pre[:, n, n] * R + self.gamma[:, n, n] * NextV - Vio[:, :, n]
        E *= self.beta[:, n, n]

        E = E - E.mean(axis=2, keepdims=True) if norm else E
        return E

    @partial(jit, static_argnums=0)
    def NextVioa(self, X, Xisa=None, Bios=None, Vio=None,
                 Tioo=None, Rio=None, Rioa=None):
        """Policy-average next value for agent i, obs o, action a."""
        Xisa = self.Xisa(X) if Xisa is None else Xisa
        Bios = self.fast_Bios(X) if Bios is None else Bios
        Vio = self.Vio(X, Rio=Rio, Tioo=Tioo, Bios=Bios, Xisa=Xisa, Rioa=Rioa) \
            if Vio is None else Vio

        i = 0; a = 1; s = 2; s_ = 3; o = 4; o_ = 5
        j2k = list(range(6, 6 + self.N - 1))
        b2d = list(range(6 + self.N - 1, 6 + self.N - 1 + self.N))
        e2f = list(range(5 + 2 * self.N, 5 + 2 * self.N + self.N - 1))

        sumsis = [[j2k[l], s, e2f[l]] for l in range(self.N - 1)]
        otherY = list(it.chain(*zip((self.N - 1) * [Xisa], sumsis)))

        args = [self.Omega, [i] + j2k + [a] + b2d + e2f,
                Bios, [i, o, s]] + otherY + \
               [self.T, [s] + b2d + [s_], self.O, [i, s_, o_],
                Vio, [i, o_], [i, o, a]]
        return jnp.einsum(*args, optimize=self.opti)

## Tests

In [ ]:
from pyCRLD.Environments.SocialDilemmaJ import SocialDilemmaJ

# SocialDilemmaJ uses identity observation (full observability),
# so this tests the PO code path with Q=Z=1
env = SocialDilemmaJ(R=3., T=5., S=0., P=1.)
agent = POstratACJ(env, learning_rates=0.1, discount_factors=0.9, choice_intensities=1.0)

X = agent.zero_intelligence_policy()  # shape (N, Q, M)
assert X.shape == (2, 1, 2)

rpe = agent.RPEioa(X)
assert rpe.shape == (2, 1, 2)  # (N, Q, M)

In [ ]:
key = jax.random.PRNGKey(3)
X0 = agent.random_softmax_policy(key=key)
traj, fpr = agent.trajectory(X0, Tmax=20)
assert traj.shape == (20, 2, 1, 2)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()